# Heatmap ETHICS Gender — Chart 06–13

Notebook ini merender langsung PNG heatmap 06–08 dan 10–13 ke `figures/`. Semua panel disusun vertikal agar angka sel dapat dibaca.

In [1]:
from __future__ import annotations

import hashlib
import sys
from pathlib import Path

import pandas as pd
REPO_ROOT = Path(r'D:/TugasAkhir')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.analysis import render_ethics_gender_accuracy_standardized as engine

NOTEBOOK_DIR = REPO_ROOT / 'notebooks' / 'gender' / 'ethics accuracy analysis'
FIGURE_DIR = NOTEBOOK_DIR / 'figures'

# Ubah angka tetap ini bila ukuran perlu direvisi lagi.
FONT_SIZES = {
    'Judul utama': 30,
    'Judul panel': 26,
    'Label sumbu X/Y': 22,
    'Tick colorbar': 22,
    'Label colorbar': 25,
    'Angka sel': 26,
}
FONT_SIZES

{'Judul utama': 30,
 'Judul panel': 26,
 'Label sumbu X/Y': 22,
 'Tick colorbar': 22,
 'Label colorbar': 25,
 'Angka sel': 26}

In [2]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def save_png(fig, figure_dir: Path, stem: str):
    figure_dir.mkdir(parents=True, exist_ok=True)
    path = figure_dir / f'{stem}.png'
    fig.savefig(path, facecolor=engine.BACKGROUND, bbox_inches='tight')
    engine.plt.close(fig)
    return [path]

engine.TITLE_SIZE = FONT_SIZES['Judul utama']
engine.PANEL_TITLE_SIZE = FONT_SIZES['Judul panel']
engine.save_figure = save_png

original_style_heatmap = engine._style_heatmap
def style_heatmap_large(ax, rotation=42):
    original_style_heatmap(ax, rotation=rotation)
    ax.tick_params(axis='x', labelsize=FONT_SIZES['Label sumbu X/Y'])
    ax.tick_params(axis='y', labelsize=FONT_SIZES['Label sumbu X/Y'])
    for text in ax.texts:
        text.set_fontsize(FONT_SIZES['Angka sel'])
        text.set_fontweight('bold' if '*' in text.get_text() else 'normal')
engine._style_heatmap = style_heatmap_large

original_heatmap = engine.sns.heatmap
def heatmap_large(*args, **kwargs):
    annotation = dict(kwargs.get('annot_kws', {}))
    annotation.update({'fontsize': FONT_SIZES['Angka sel'], 'fontweight': 'bold'})
    kwargs['annot_kws'] = annotation
    ax = kwargs.get('ax')
    result = original_heatmap(*args, **kwargs)
    for text in ax.texts:
        text.set_fontsize(FONT_SIZES['Angka sel'])
        text.set_fontweight('bold' if '*' in text.get_text() else 'normal')
    colorbar = ax.collections[0].colorbar
    colorbar.ax.tick_params(labelsize=FONT_SIZES['Tick colorbar'])
    colorbar.ax.yaxis.label.set_size(FONT_SIZES['Label colorbar'])
    return result
engine.sns.heatmap = heatmap_large

def plot_summary_vertical(summary, metric, subsets, cmap, title, stem, colorbar_label):
    fig, axes = engine.plt.subplots(len(subsets), 1, figsize=(18, 8.7 * len(subsets)), squeeze=False, constrained_layout=True)
    for ax, subset in zip(axes.flat, subsets, strict=True):
        matrix = engine._matrix(summary[summary['subset'].eq(subset)], metric) * 100
        engine.sns.heatmap(matrix, annot=True, fmt='.1f', cmap=cmap, vmin=0, vmax=100, linewidths=0.5, cbar_kws={'label': colorbar_label}, ax=ax)
        ax.set_title(engine.SUBSET_LABELS[subset], fontsize=FONT_SIZES['Judul panel'], fontweight='bold')
        ax.set_xlabel('')
        ax.set_ylabel('')
        engine._style_heatmap(ax)
    fig.suptitle(title, fontsize=FONT_SIZES['Judul utama'], fontweight='bold')
    return engine.save_figure(fig, FIGURE_DIR, stem)

def plot_effect_vertical(paired, figure_dir: Path):
    specs = [
        ('coverage', 'Commit Rate', '10_ethics_coverage_effect_holm_heatmap'),
        ('conditional_accuracy', 'Akurasi', '11_ethics_conditional_accuracy_effect_holm_heatmap'),
        ('effective_success', 'Keberhasilan efektif', '12_ethics_effective_success_effect_holm_heatmap'),
        ('stance_wrong', 'Distribusi keputusan', '13_ethics_stance_wrong_effect_holm_heatmap'),
    ]
    paths = []
    for outcome, outcome_label, stem in specs:
        frame = paired[paired['outcome'].eq(outcome)].copy()
        subsets = [subset for subset in engine.SUBSET_ORDER if subset in set(frame['subset'])]
        fig, axes = engine.plt.subplots(len(subsets), 1, figsize=(18, 8.7 * len(subsets)), squeeze=False, constrained_layout=True)
        limit = max(1.0, float(engine.np.nanmax(engine.np.abs(frame['risk_difference_pp']))))
        for ax, subset in zip(axes.flat, subsets, strict=True):
            subset_frame = frame[frame['subset'].eq(subset)]
            engine.sns.heatmap(engine._effect_value_matrix(subset_frame), annot=engine._effect_annot_matrix(subset_frame), fmt='', cmap='RdBu_r', center=0, vmin=-limit, vmax=limit, linewidths=0.5, cbar_kws={'label': 'Risk difference (percentage points)'}, ax=ax)
            ax.set_title(engine.SUBSET_LABELS[subset], fontsize=FONT_SIZES['Judul panel'], fontweight='bold')
            ax.set_xlabel('')
            ax.set_ylabel('')
            engine._style_heatmap(ax)
        fig.suptitle(f'ETHICS Gender: {outcome_label}', fontsize=FONT_SIZES['Judul utama'], fontweight='bold')
        paths.extend(engine.save_figure(fig, figure_dir, stem))
    return paths

tables = engine._load_source_tables(engine.SOURCE_BUNDLE)
derived = engine.derive_tables(tables)
engine.configure_plot_style()

rendered = []
rendered += plot_summary_vertical(derived['summary'], 'accuracy_committed', engine.VALID_SUBSETS, 'RdYlGn', 'ETHICS Gender: Akurasi per Subset', '06_ethics_accuracy_committed_valid_subsets_heatmap', 'Accuracy committed (%)')
rendered += plot_summary_vertical(derived['summary'], 'effective_success_rate_nonmissing', engine.VALID_SUBSETS, 'RdYlGn', 'ETHICS Gender: Keberhasilan Efektif per Subset', '07_ethics_effective_success_valid_subsets_heatmap', 'Effective success (%)')
rendered += plot_summary_vertical(derived['summary'], 'pct_answer_wrong', engine.INVALID_STANCE_SUBSETS, 'RdYlBu_r', 'ETHICS Gender: Distribusi Keputusan Deontology dan Virtue', '08_ethics_stance_wrong_construct_invalid_heatmap', 'Stance wrong (%)')
rendered += plot_effect_vertical(derived['paired_primary'], FIGURE_DIR)

index_path = NOTEBOOK_DIR / 'output_index.csv'
index = pd.read_csv(index_path)
for path in rendered:
    index.loc[index['file'].eq(path.name), 'sha256'] = sha256_file(path)
index.to_csv(index_path, index=False)

assert all(path.suffix == '.png' for path in rendered)
assert not list(FIGURE_DIR.glob('*.svg'))
assert all(index.loc[index['file'].eq(path.name), 'sha256'].iloc[0] == sha256_file(path) for path in rendered)
print('PNG langsung diperbarui:', len(rendered))

PNG langsung diperbarui: 7
